In [1]:
import sys

sys.path.insert(0, "../")

import pandas as pd
from mdu.eval.table_analysis_utils import (
    transform_by_tasks,
    select_composite_and_components,
    check_composite_dominance,
    compute_average_ranks,
    analyze_composite_pareto_performance,
)
from notebooks.table_utils import with_avg_row, mean_pm_std, fmt_valvar
from mdu.unc.constants import OTTarget, ScalingType

# Set pandas display options to show all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

%load_ext autoreload
%autoreload 2

In [56]:
target_distr = OTTarget.BETA.value
scaling_type = ScalingType.FEATURE_WISE.value
grid_size = 0 # 0 5
n_targets_multiplier = 1


# df = pd.read_csv(
#     f"../resources/refactored/benchmark_entropic_target_{target_distr}_eps_0.5_scaling_type_{scaling_type}_iters_1000_tol_1e-06_rs_42_grid_size_{grid_size}_n_targets_multiplier_{n_targets_multiplier}.csv"
# )

df = pd.read_csv(
    f"../resources/refactored/calibration_001_entropic_target_{target_distr}_eps_0.5_scaling_type_{scaling_type}_iters_1000_tol_1e-06_rs_42_grid_size_{grid_size}_n_targets_multiplier_{n_targets_multiplier}.csv"
)

# df = pd.read_csv(
#     f"./additive_baseline.csv"
# )

In [57]:
df.head(10)

,ind_dataset,ood_dataset,measure,uncertainty_type,gname,risk_type,gt_approximation,pred_approximation,ensemble_group,problem_type,roc_auc,average_precision,accuracy,aurc,acc_cov_auc,coverage_at_1pct_error,coverage_at_2pct_error,coverage_at_5pct_error,n_ind_samples,n_ood_samples,n_correct,n_incorrect,ensemble_accuracy
0,cifar10,cifar100,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,0,ood_detection,0.912015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.963750
1,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,0,misclassification_detection,0.944999,0.381728,0.963750,NaN,NaN,NaN,NaN,NaN,7200,NaN,6939.0,261.0,0.963750
2,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,0,selective_prediction,NaN,NaN,0.963750,0.002928,0.996933,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.963750
3,cifar10,cifar100,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,1,ood_detection,0.910690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.964306
4,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,1,misclassification_detection,0.944610,0.374033,0.964306,NaN,NaN,NaN,NaN,NaN,7200,NaN,6943.0,257.0,0.964306
5,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,1,selective_prediction,NaN,NaN,0.964306,0.002930,0.996931,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.964306
6,cifar10,cifar100,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,2,ood_detection,0.911888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.963333
7,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,2,misclassification_detection,0.943641,0.360288,0.963333,NaN,NaN,NaN,NaN,NaN,7200,NaN,6936.0,264.0,0.963333
8,cifar10,cifar10,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,2,selective_prediction,NaN,NaN,0.963333,0.003023,0.996838,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.963333
9,cifar10,cifar100,Risk_LogScore_TotalRisk_outer_outer,Risk,LogScore,TotalRisk,outer,outer,3,ood_detection,0.911436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.964167


In [58]:
print(df.ind_dataset.unique())
print(df.ood_dataset.unique())

['cifar10' 'cifar100' 'tiny_imagenet']
['cifar100' 'cifar10' 'tiny_imagenet' 'svhn' 'imagenet_a' 'imagenet_o'
 'imagenet_r']


In [59]:
include_std = True

if not include_std:
    transformed_df = transform_by_tasks(df, include_std=include_std)
else:
    transformed_df, std_df = transform_by_tasks(df, include_std=include_std)

In [60]:
transformed_df

measure                                  R_b 1 (Logscore)  R_e 1 1 (Logscore)  \
ind_dataset   eval                                                              
cifar10       cifar10 [miscls]                   0.942267            0.940430   
              cifar10 [selective]                0.996818            0.996843   
              cifar100 [ood]                     0.916906            0.904730   
              svhn [ood]                         0.962992            0.942616   
              tiny_imagenet [ood]                0.911364            0.895731   
cifar100      cifar10 [ood]                      0.773270            0.724548   
              cifar100 [miscls]                  0.845094            0.818006   
              cifar100 [selective]               0.915943            0.909667   
              svhn [ood]                         0.870120            0.755848   
              tiny_imagenet [ood]                0.809888            0.999852   
tiny_imagenet tiny_imagenet [miscls]             0.844739            0.813053   
              tiny_imagenet [selective]          0.888853            0.879249   
              imagenet_a [ood]                   0.835350            0.781130   
              imagenet_r [ood]                   0.825339            0.774366   
              imagenet_o [ood]                   0.724312            0.752832   

measure                                  R_t 1 1 (Logscore)  \
ind_dataset   eval                                            
cifar10       cifar10 [miscls]                     0.943244   
              cifar10 [selective]                  0.996864   
              cifar100 [ood]                       0.911507   
              svhn [ood]                           0.956506   
              tiny_imagenet [ood]                  0.903615   
cifar100      cifar10 [ood]                        0.774023   
              cifar100 [miscls]                    0.853064   
              cifar100 [selective]                 0.918425   
              svhn [ood]                           0.867727   
              tiny_imagenet [ood]                  0.999997   
tiny_imagenet tiny_imagenet [miscls]               0.850815   
              tiny_imagenet [selective]            0.890910   
              imagenet_a [ood]                     0.846305   
              imagenet_r [ood]                     0.836572   
              imagenet_o [ood]                     0.753524   

measure                                  composite eat logscore outer outer + m  \
ind_dataset   eval                                                                
cifar10       cifar10 [miscls]                                         0.941799   
              cifar10 [selective]                                      0.996847   
              cifar100 [ood]                                           0.917602   
              svhn [ood]                                               0.954178   
              tiny_imagenet [ood]                                      0.912771   
cifar100      cifar10 [ood]                                            0.757150   
              cifar100 [miscls]                                        0.839752   
              cifar100 [selective]                                     0.910792   
              svhn [ood]                                               0.882455   
              tiny_imagenet [ood]                                      0.815082   
tiny_imagenet tiny_imagenet [miscls]                                   0.836517   
              tiny_imagenet [selective]                                0.879883   
              imagenet_a [ood]                                         0.844182   
              imagenet_r [ood]                                         0.827128   
              imagenet_o [ood]                                         0.755068   

measure                                  mahalanobis  
ind_dataset   eval                                    
cifar10       cifar10 [miscl

In [61]:
std_df

R_b 1 (Logscore)  R_e 1 1 (Logscore)  \
ind_dataset   eval                                                              
cifar10       cifar10 [miscls]                   0.001585            0.003253   
              cifar10 [selective]                0.000049            0.000145   
              cifar100 [ood]                     0.001002            0.000337   
              svhn [ood]                         0.002252            0.012131   
              tiny_imagenet [ood]                0.001037            0.000245   
cifar100      cifar10 [ood]                      0.002164            0.001497   
              cifar100 [miscls]                  0.002761            0.002878   
              cifar100 [selective]               0.000890            0.001027   
              svhn [ood]                         0.006667            0.013424   
              tiny_imagenet [ood]                0.000847            0.000038   
tiny_imagenet tiny_imagenet [miscls]             0.002899            0.001207   
              tiny_imagenet [selective]          0.000570            0.001284   
              imagenet_a [ood]                   0.002358            0.003696   
              imagenet_r [ood]                   0.003457            0.003244   
              imagenet_o [ood]                   0.003811            0.002095   

                                         R_t 1 1 (Logscore)  \
ind_dataset   eval                                            
cifar10       cifar10 [miscls]                     0.002413   
              cifar10 [selective]                  0.000086   
              cifar100 [ood]                       0.000599   
              svhn [ood]                           0.007015   
              tiny_imagenet [ood]                  0.000583   
cifar100      cifar10 [ood]                        0.001598   
              cifar100 [miscls]                    0.002791   
              cifar100 [selective]                 0.000848   
              svhn [ood]                           0.005873   
              tiny_imagenet [ood]                  0.000004   
tiny_imagenet tiny_imagenet [miscls]               0.002244   
              tiny_imagenet [selective]            0.000442   
              imagenet_a [ood]                     0.002370   
              imagenet_r [ood]                     0.001728   
              imagenet_o [ood]                     0.001961   

                                         composite eat logscore outer outer + m  \
ind_dataset   eval                                                                
cifar10       cifar10 [miscls]                                         0.003533   
              cifar10 [selective]                                      0.000121   
              cifar100 [ood]                                           0.003457   
              svhn [ood]                                               0.003125   
              tiny_imagenet [ood]                                      0.002282   
cifar100      cifar10 [ood]                                            0.002744   
              cifar100 [miscls]                                        0.007456   
              cifar100 [selective]                                     0.002150   
              svhn [ood]                                               0.006657   
              tiny_imagenet [ood]                                      0.033740   
tiny_imagenet tiny_imagenet [miscls]                                   0.003682   
              tiny_imagenet [selective]                                0.001484   
              imagenet_a [ood]                                         0.002563   
              imagenet_r [ood]                                         0.001946   
              imagenet_o [ood]                                         0.002244   

                                         mahalanobis  
ind_dataset   eval                                    
cifar10       cifar10 [miscls]              0.002597  
              

In [62]:
latex_df = fmt_valvar(
    transformed_df[['composite eat logscore outer outer + m']],
    std_df[['composite eat logscore outer outer + m']],
    mean_decimals=3,
    std_decimals=3,
    bold=False,
    underline=False,
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lll}
\toprule
 & measure & composite eat logscore outer outer + m \\
ind_dataset & eval &  \\
\midrule
\multirow[t]{5}{*}{cifar10} & cifar10 [miscls] & \valvar{0.942}{.004} \\
 & cifar10 [selective] & \valvar{0.997}{.000} \\
 & cifar100 [ood] & \valvar{0.918}{.003} \\
 & svhn [ood] & \valvar{0.954}{.003} \\
 & tiny_imagenet [ood] & \valvar{0.913}{.002} \\
\cline{1-3}
\multirow[t]{5}{*}{cifar100} & cifar10 [ood] & \valvar{0.757}{.003} \\
 & cifar100 [miscls] & \valvar{0.840}{.007} \\
 & cifar100 [selective] & \valvar{0.911}{.002} \\
 & svhn [ood] & \valvar{0.882}{.007} \\
 & tiny_imagenet [ood] & \valvar{0.815}{.034} \\
\cline{1-3}
\multirow[t]{5}{*}{tiny_imagenet} & tiny_imagenet [miscls] & \valvar{0.837}{.004} \\
 & tiny_imagenet [selective] & \valvar{0.880}{.001} \\
 & imagenet_a [ood] & \valvar{0.844}{.003} \\
 & imagenet_r [ood] & \valvar{0.827}{.002} \\
 & imagenet_o [ood] & \valvar{0.755}{.002} \\
\cline{1-3}
\bottomrule
\end{tabular}



In [9]:
from configs.interesting_compositions import INTERESTING_COMPOSITIONS

for k in INTERESTING_COMPOSITIONS.keys():
    print(k)

COMPOSITE EAT LOGSCORE OUTER OUTER + M


In [10]:
measure_name = "COMPOSITE EAT LOGSCORE OUTER OUTER + M"

res_df = select_composite_and_components(transformed_df, measure_name)
res_df_std = select_composite_and_components(std_df, measure_name)

res_df_with_dominance = check_composite_dominance(res_df)

composite_pareto_results = analyze_composite_pareto_performance(
    transformed_df,
    INTERESTING_COMPOSITIONS,
    do_for_each_measure=True,
    different_only=False,
)

In [21]:
latex_df = fmt_valvar(
    res_df[['composite eat logscore outer outer + m']],
    res_df_std[['composite eat logscore outer outer + m']],
    mean_decimals=3,
    std_decimals=3,
    bold=False,
    underline=False,
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lll}
\toprule
 & measure & composite eat logscore outer outer + m \\
ind_dataset & eval &  \\
\midrule
\multirow[t]{5}{*}{cifar10} & cifar10 [miscls] & \valvar{0.944}{.002} \\
 & cifar10 [selective] & \valvar{0.997}{.000} \\
 & cifar100 [ood] & \valvar{0.918}{.001} \\
 & svhn [ood] & \valvar{0.957}{.005} \\
 & tiny_imagenet [ood] & \valvar{0.912}{.001} \\
\cline{1-3}
\multirow[t]{5}{*}{cifar100} & cifar10 [ood] & \valvar{0.765}{.001} \\
 & cifar100 [miscls] & \valvar{0.849}{.003} \\
 & cifar100 [selective] & \valvar{0.916}{.001} \\
 & svhn [ood] & \valvar{0.870}{.006} \\
 & tiny_imagenet [ood] & \valvar{1.000}{.000} \\
\cline{1-3}
\multirow[t]{5}{*}{tiny_imagenet} & tiny_imagenet [miscls] & \valvar{0.847}{.002} \\
 & tiny_imagenet [selective] & \valvar{0.886}{.001} \\
 & imagenet_a [ood] & \valvar{0.847}{.002} \\
 & imagenet_r [ood] & \valvar{0.835}{.001} \\
 & imagenet_o [ood] & \valvar{0.760}{.002} \\
\cline{1-3}
\bottomrule
\end{tabular}



In [11]:
composite_pareto_results

{'COMPOSITE EAT LOGSCORE OUTER OUTER + M': {'pareto_count': 87,
  'total_problems': 105,
  'pareto_percentage': 82.85714285714286,
  'average_pareto_depth': np.float64(0.2),
  'median_pareto_depth': np.float64(0.0),
  'individual_measures': {'R_e 1 1 (Logscore)': {'pareto_count': 0,
    'total_problems': 105,
    'pareto_percentage': 0.0,
    'average_pareto_depth': np.float64(1.819047619047619),
    'median_pareto_depth': np.float64(2.0)},
   'R_t 1 1 (Logscore)': {'pareto_count': 69,
    'total_problems': 105,
    'pareto_percentage': 65.71428571428571,
    'average_pareto_depth': np.float64(0.38095238095238093),
    'median_pareto_depth': np.float64(0.0)},
   'R_b 1 (Logscore)': {'pareto_count': 23,
    'total_problems': 105,
    'pareto_percentage': 21.904761904761905,
    'average_pareto_depth': np.float64(0.9333333333333333),
    'median_pareto_depth': np.float64(1.0)},
   'mahalanobis': {'pareto_count': 0,
    'total_problems': 105,
    'pareto_percentage': 0.0,
    'average_par

In [12]:
display(res_df_with_dominance)
display(res_df_std)

print("==" * 100)
print("Pareto Percentages:")

for k, result in composite_pareto_results.items():
    if measure_name == k:
        indv_ = result["individual_measures"]
        print(measure_name, result["pareto_percentage"])
        for name, el in indv_.items():
            print(name, el["pareto_percentage"])

measure                                  R_e 1 1 (Logscore)  \
ind_dataset   eval                                            
cifar10       cifar10 [miscls]                     0.940430   
              cifar10 [selective]                  0.996843   
              cifar100 [ood]                       0.904730   
              svhn [ood]                           0.942616   
              tiny_imagenet [ood]                  0.895731   
cifar100      cifar10 [ood]                        0.724548   
              cifar100 [miscls]                    0.818006   
              cifar100 [selective]                 0.909667   
              svhn [ood]                           0.755848   
              tiny_imagenet [ood]                  0.999852   
tiny_imagenet tiny_imagenet [miscls]               0.813053   
              tiny_imagenet [selective]            0.879249   
              imagenet_a [ood]                     0.781130   
              imagenet_r [ood]                     0.774366   
              imagenet_o [ood]                     0.752832   

measure                                  R_t 1 1 (Logscore)  R_b 1 (Logscore)  \
ind_dataset   eval                                                              
cifar10       cifar10 [miscls]                     0.943244          0.942267   
              cifar10 [selective]                  0.996864          0.996818   
              cifar100 [ood]                       0.911507          0.916906   
              svhn [ood]                           0.956506          0.962992   
              tiny_imagenet [ood]                  0.903615          0.911364   
cifar100      cifar10 [ood]                        0.774023          0.773270   
              cifar100 [miscls]                    0.853064          0.845094   
              cifar100 [selective]                 0.918425          0.915943   
              svhn [ood]                           0.867727          0.870120   
              tiny_imagenet [ood]                  0.999997          0.809888   
tiny_imagenet tiny_imagenet [miscls]               0.850815          0.844739   
              tiny_imagenet [selective]            0.890910          0.888853   
              imagenet_a [ood]                     0.846305          0.835350   
              imagenet_r [ood]                     0.836572          0.825339   
              imagenet_o [ood]                     0.753524          0.724312   

measure                                  mahalanobis  \
ind_dataset   eval                                     
cifar10       cifar10 [miscls]              0.927621   
              cifar10 [selective]           0.996266   
              cifar100 [ood]                0.912238   
              svhn [ood]                    0.934311   
              tiny_imagenet [ood]           0.910273   
cifar100      cifar10 [ood]                 0.534822   
              cifar100 [miscls]             0.573908   
              cifar100 [selective]          0.810550   
              svhn [ood]                    0.678832   
              tiny_imagenet [ood]           0.622941   
tiny_imagenet tiny_imagenet [miscls]        0.416754   
              tiny_imagenet [selective]     0.659777   
              imagenet_a [ood]              0.440974   
              imagenet_r [ood]              0.404755   
              imagenet_o [ood]              0.512686   

measure                                  composite eat logscore outer outer + m  \
ind_dataset   eval                                                                
cifar10       cifar10 [miscls]                                         0.943972   
              cifar10 [selective]                                      0.996923   
              cifar100 [ood]                                           0.917981   
              svhn [ood]                                               0.957479   
              tiny_imagenet [ood]                                      0.912092   
cifar100      ci

R_e 1 1 (Logscore)  \
ind_dataset   eval                                            
cifar10       cifar10 [miscls]                     0.003253   
              cifar10 [selective]                  0.000145   
              cifar100 [ood]                       0.000337   
              svhn [ood]                           0.012131   
              tiny_imagenet [ood]                  0.000245   
cifar100      cifar10 [ood]                        0.001497   
              cifar100 [miscls]                    0.002878   
              cifar100 [selective]                 0.001027   
              svhn [ood]                           0.013424   
              tiny_imagenet [ood]                  0.000038   
tiny_imagenet tiny_imagenet [miscls]               0.001207   
              tiny_imagenet [selective]            0.001284   
              imagenet_a [ood]                     0.003696   
              imagenet_r [ood]                     0.003244   
              imagenet_o [ood]                     0.002095   

                                         R_t 1 1 (Logscore)  R_b 1 (Logscore)  \
ind_dataset   eval                                                              
cifar10       cifar10 [miscls]                     0.002413          0.001585   
              cifar10 [selective]                  0.000086          0.000049   
              cifar100 [ood]                       0.000599          0.001002   
              svhn [ood]                           0.007015          0.002252   
              tiny_imagenet [ood]                  0.000583          0.001037   
cifar100      cifar10 [ood]                        0.001598          0.002164   
              cifar100 [miscls]                    0.002791          0.002761   
              cifar100 [selective]                 0.000848          0.000890   
              svhn [ood]                           0.005873          0.006667   
              tiny_imagenet [ood]                  0.000004          0.000847   
tiny_imagenet tiny_imagenet [miscls]               0.002244          0.002899   
              tiny_imagenet [selective]            0.000442          0.000570   
              imagenet_a [ood]                     0.002370          0.002358   
              imagenet_r [ood]                     0.001728          0.003457   
              imagenet_o [ood]                     0.001961          0.003811   

                                         mahalanobis  \
ind_dataset   eval                                     
cifar10       cifar10 [miscls]              0.002597   
              cifar10 [selective]           0.000044   
              cifar100 [ood]                0.001507   
              svhn [ood]                    0.005441   
              tiny_imagenet [ood]           0.001056   
cifar100      cifar10 [ood]                 0.004189   
              cifar100 [miscls]             0.006341   
              cifar100 [selective]          0.001328   
              svhn [ood]                    0.033548   
              tiny_imagenet [ood]           0.005384   
tiny_imagenet tiny_imagenet [miscls]        0.003688   
              tiny_imagenet [selective]     0.005281   
              imagenet_a [ood]              0.009082   
              imagenet_r [ood]              0.007956   
              imagenet_o [ood]              0.004898   

                                         composite eat logscore outer outer + m  
ind_dataset   eval                                                               
cifar10       cifar10 [miscls]                                         0.001925  
              cifar10 [selective]                                      0.000100  
              cifar100 [ood]                                           0.000899  
              svhn [ood]                                               0.004980  
              tiny_imagenet [ood]                                      0.000620  
cifar100      cifar10 [ood]                                     

Pareto Percentages:
COMPOSITE EAT LOGSCORE OUTER OUTER + M 82.85714285714286
R_e 1 1 (Logscore) 0.0
R_t 1 1 (Logscore) 65.71428571428571
R_b 1 (Logscore) 21.904761904761905
mahalanobis 0.0


In [13]:
# assume your df has MultiIndex with level names ['ind_dataset', 'eval']
lvl = res_df_with_dominance.index.get_level_values("eval")

df_ood = res_df_with_dominance[lvl.str.contains(r"\[ood\]")]
df_miscls = res_df_with_dominance[lvl.str.contains(r"\[miscls\]")]
df_selective = res_df_with_dominance[lvl.str.contains(r"\[selective\]")]

# assume your df has MultiIndex with level names ['ind_dataset', 'eval']
lvl_std = res_df_std.index.get_level_values("eval")

df_ood_std = res_df_std[lvl_std.str.contains(r"\[ood\]")]
df_miscls_std = res_df_std[lvl_std.str.contains(r"\[miscls\]")]
df_selective_std = res_df_std[lvl_std.str.contains(r"\[selective\]")]

df_ood = df_ood.drop(
    columns=[
        "if_dominates_100%",
        "if_dominates_75%",
        "if_dominates_50%",
        "beats_worst_component",
    ]
)

df_miscls = df_miscls.drop(
    columns=[
        "if_dominates_100%",
        "if_dominates_75%",
        "if_dominates_50%",
        "beats_worst_component",
    ]
)

df_selective = df_selective.drop(
    columns=[
        "if_dominates_100%",
        "if_dominates_75%",
        "if_dominates_50%",
        "beats_worst_component",
    ]
)

In [ ]:
col_order = [
    "composite eat logscore outer outer + m",
    "R_e 1 1 (Logscore)",
    "R_t 1 1 (Logscore)",
    "R_b 1 (Logscore)",
    "mahalanobis",
]

In [16]:
df_ood = with_avg_row(df_ood, label=("AVG", "[all rows]"))[col_order]
df_ood_std = with_avg_row(df_ood_std, label=("AVG", "[all rows]"))[col_order]

latex_df = mean_pm_std(
    df_mean=df_ood,
    df_std=df_ood_std,
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lll}
\toprule
 &  & composite eat logscore outer outer + m \\
ind_dataset & eval &  \\
\midrule
\multirow[t]{3}{*}{cifar10} & cifar100 [ood] & $0.918 \pm 0.001$ \\
 & svhn [ood] & $0.957 \pm 0.005$ \\
 & tiny_imagenet [ood] & $0.912 \pm 0.001$ \\
\cline{1-3}
\multirow[t]{3}{*}{cifar100} & cifar10 [ood] & $0.765 \pm 0.001$ \\
 & svhn [ood] & $0.870 \pm 0.006$ \\
 & tiny_imagenet [ood] & $1.000 \pm 0.000$ \\
\cline{1-3}
\multirow[t]{3}{*}{tiny_imagenet} & imagenet_a [ood] & $0.847 \pm 0.002$ \\
 & imagenet_r [ood] & $0.835 \pm 0.001$ \\
 & imagenet_o [ood] & $0.760 \pm 0.002$ \\
\cline{1-3}
AVG & [all rows] & $0.874 \pm 0.002$ \\
\cline{1-3}
\bottomrule
\end{tabular}



In [16]:
df_miscls = with_avg_row(df_miscls, label=("AVG", "[all rows]"))[col_order]
df_miscls_std = with_avg_row(df_miscls_std, label=("AVG", "[all rows]"))[col_order]

latex_df = fmt_valvar(
    df_miscls,
    df_miscls_std,
    mean_decimals=3,
    std_decimals=3,
    bold=False,
    underline=False,
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lllllll}
\toprule
 & measure & composite eat logscore outer outer + m & R_e 1 1 (Logscore) & R_t 1 1 (Logscore) & R_b 1 (Logscore) & mahalanobis \\
ind_dataset & eval &  &  &  &  &  \\
\midrule
cifar10 & cifar10 [miscls] & \valvar{0.944}{.002} & \valvar{0.940}{.003} & \valvar{0.943}{.002} & \valvar{0.942}{.002} & \valvar{0.928}{.003} \\
\cline{1-7}
cifar100 & cifar100 [miscls] & \valvar{0.849}{.003} & \valvar{0.818}{.003} & \valvar{0.853}{.003} & \valvar{0.845}{.003} & \valvar{0.574}{.006} \\
\cline{1-7}
tiny_imagenet & tiny_imagenet [miscls] & \valvar{0.847}{.002} & \valvar{0.813}{.001} & \valvar{0.851}{.002} & \valvar{0.845}{.003} & \valvar{0.417}{.004} \\
\cline{1-7}
AVG & [all rows] & \valvar{0.880}{.002} & \valvar{0.857}{.002} & \valvar{0.882}{.002} & \valvar{0.877}{.002} & \valvar{0.639}{.004} \\
\cline{1-7}
\bottomrule
\end{tabular}



In [16]:
df_selective = with_avg_row(df_selective, label=("AVG", "[all rows]"))[col_order]
df_selective_std = with_avg_row(df_selective_std, label=("AVG", "[all rows]"))[
    col_order
]

latex_df = fmt_valvar(
    df_selective,
    df_selective_std,
    mean_decimals=3,
    std_decimals=3,
    bold=False,
    underline=False,
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lllllll}
\toprule
 & measure & composite eat logscore outer outer + m & R_e 1 1 (Logscore) & R_t 1 1 (Logscore) & R_b 1 (Logscore) & mahalanobis \\
ind_dataset & eval &  &  &  &  &  \\
\midrule
cifar10 & cifar10 [selective] & \valvar{0.997}{.000} & \valvar{0.997}{.000} & \valvar{0.997}{.000} & \valvar{0.997}{.000} & \valvar{0.996}{.000} \\
\cline{1-7}
cifar100 & cifar100 [selective] & \valvar{0.916}{.001} & \valvar{0.910}{.001} & \valvar{0.918}{.001} & \valvar{0.916}{.001} & \valvar{0.811}{.001} \\
\cline{1-7}
tiny_imagenet & tiny_imagenet [selective] & \valvar{0.886}{.001} & \valvar{0.879}{.001} & \valvar{0.891}{.000} & \valvar{0.889}{.001} & \valvar{0.660}{.005} \\
\cline{1-7}
AVG & [all rows] & \valvar{0.933}{.001} & \valvar{0.929}{.001} & \valvar{0.935}{.000} & \valvar{0.934}{.001} & \valvar{0.822}{.002} \\
\cline{1-7}
\bottomrule
\end{tabular}



In [17]:
import pandas as pd
import numpy as np


def highlight_best_and_second(s, best="#27ef56", second="#3908ed"):
    # rank 1 = largest; ties share the same rank
    r = s.rank(method="min", ascending=False)
    out = []
    for val, ri in zip(s, r):
        if pd.isna(val):
            out.append("")
        elif ri == 1:
            out.append(f"background-color: {best}; font-weight: bold")
        elif ri == 2:
            out.append(f"background-color: {second}")
        else:
            out.append("")
    return out

In [18]:
[el for el in transformed_df.columns]

['R_b 1 (Brier)',
 'R_b 1 (Logscore)',
 'R_b 1 (Spherical)',
 'R_b 1 (Zero-one)',
 'R_b 2 (Brier)',
 'R_b 2 (Logscore)',
 'R_b 2 (Spherical)',
 'R_b 2 (Zero-one)',
 'R_b 3 (Brier)',
 'R_b 3 (Logscore)',
 'R_b 3 (Spherical)',
 'R_b 3 (Zero-one)',
 'R_e 1 1 (Brier)',
 'R_e 1 1 (Logscore)',
 'R_e 1 1 (Spherical)',
 'R_e 1 1 (Zero-one)',
 'R_e 1 2 (Brier)',
 'R_e 1 2 (Logscore)',
 'R_e 1 2 (Spherical)',
 'R_e 1 2 (Zero-one)',
 'R_e 1 3 (Brier)',
 'R_e 1 3 (Logscore)',
 'R_e 1 3 (Spherical)',
 'R_e 1 3 (Zero-one)',
 'R_e 2 1 (Brier)',
 'R_e 2 1 (Logscore)',
 'R_e 2 1 (Spherical)',
 'R_e 2 1 (Zero-one)',
 'R_e 2 2 (Brier)',
 'R_e 2 2 (Logscore)',
 'R_e 2 2 (Spherical)',
 'R_e 2 2 (Zero-one)',
 'R_e 2 3 (Brier)',
 'R_e 2 3 (Logscore)',
 'R_e 2 3 (Spherical)',
 'R_e 2 3 (Zero-one)',
 'R_e 3 1 (Brier)',
 'R_e 3 1 (Logscore)',
 'R_e 3 1 (Spherical)',
 'R_e 3 1 (Zero-one)',
 'R_e 3 2 (Brier)',
 'R_e 3 2 (Logscore)',
 'R_e 3 2 (Spherical)',
 'R_e 3 2 (Zero-one)',
 'R_e 3 3 (Brier)',
 'R_e 3 3 (Log

In [30]:
# col_order = ['composite bayes all outer', 'R_b 1 (Logscore)', 'R_b 1 (Brier)', 'R_b 1 (Spherical)', 'R_b 1 (Zero-one)']
col_order = [
    "composite excess all outer outer",
    "R_e 1 1 (Logscore)",
    "R_e 1 1 (Brier)",
    "R_e 1 1 (Spherical)",
    "R_e 1 1 (Zero-one)",
]
transformed_df[col_order].style.apply(highlight_best_and_second, axis=1)

In [31]:
# col_order = ['composite bayes all inner', 'R_b 2 (Logscore)', 'R_b 2 (Brier)', 'R_b 2 (Spherical)', 'R_b 2 (Zero-one)']
col_order = [
    "composite excess all outer inner",
    "R_e 1 2 (Logscore)",
    "R_e 1 2 (Brier)",
    "R_e 1 2 (Spherical)",
    "R_e 1 2 (Zero-one)",
]
transformed_df[col_order].style.apply(highlight_best_and_second, axis=1)

In [32]:
# col_order = ['composite bayes all central', 'R_b 3 (Logscore)', 'R_b 3 (Brier)', 'R_b 3 (Spherical)', 'R_b 3 (Zero-one)']
col_order = [
    "composite excess all outer central",
    "R_e 1 3 (Logscore)",
    "R_e 1 3 (Brier)",
    "R_e 1 3 (Spherical)",
    "R_e 1 3 (Zero-one)",
]
transformed_df[col_order].style.apply(highlight_best_and_second, axis=1)

In [36]:
# col_order = ['composite bayes all outer', 'R_b 1 (Logscore)', 'R_b 1 (Brier)', 'R_b 1 (Spherical)', 'R_b 1 (Zero-one)']
col_order = [
    "composite excess all outer central",
    "R_e 1 3 (Logscore)",
    "R_e 1 3 (Brier)",
    "R_e 1 3 (Spherical)",
    "R_e 1 3 (Zero-one)",
]
latex_df = mean_pm_std(
    df_mean=with_avg_row(transformed_df[col_order], label=("AVG", "[all rows]"))[
        col_order
    ],
    df_std=with_avg_row(std_df[col_order], label=("AVG", "[all rows]"))[col_order],
)
latex_table = latex_df.to_latex(escape=False)
print(latex_table)

\begin{tabular}{lllllll}
\toprule
 &  & composite excess all outer central & R_e 1 3 (Logscore) & R_e 1 3 (Brier) & R_e 1 3 (Spherical) & R_e 1 3 (Zero-one) \\
ind_dataset & eval &  &  &  &  &  \\
\midrule
\multirow[t]{5}{*}{cifar10} & cifar10 [miscls] & $0.942 \pm 0.003$ & $0.936 \pm 0.003$ & $0.942 \pm 0.003$ & $0.942 \pm 0.003$ & $0.797 \pm 0.008$ \\
 & cifar10 [selective] & $0.997 \pm 0.000$ & $0.997 \pm 0.000$ & $0.997 \pm 0.000$ & $0.997 \pm 0.000$ & $0.983 \pm 0.002$ \\
 & cifar100 [ood] & $0.904 \pm 0.000$ & $0.903 \pm 0.001$ & $0.902 \pm 0.000$ & $0.904 \pm 0.000$ & $0.755 \pm 0.001$ \\
 & svhn [ood] & $0.941 \pm 0.011$ & $0.940 \pm 0.013$ & $0.940 \pm 0.010$ & $0.942 \pm 0.010$ & $0.825 \pm 0.038$ \\
 & tiny_imagenet [ood] & $0.895 \pm 0.000$ & $0.894 \pm 0.001$ & $0.893 \pm 0.000$ & $0.894 \pm 0.001$ & $0.752 \pm 0.002$ \\
\cline{1-7}
\multirow[t]{5}{*}{cifar100} & cifar10 [ood] & $0.710 \pm 0.002$ & $0.718 \pm 0.002$ & $0.681 \pm 0.002$ & $0.715 \pm 0.002$ & $0.689 \pm 0.00

In [24]:
transformed_df[col_order].style.apply(highlight_best_and_second, axis=1)

In [25]:
transformed_df[col_order].style.apply(highlight_best_and_second, axis=1)